# Org1 Graph Construction

In [3]:
from pathlib import Path
import duckdb

In [4]:
ORG1_PATH = (
    Path("../data/organizations/organization=Org1")
    .resolve()
    .as_posix()
    + "/*.parquet"
)

print(ORG1_PATH)

C:/Users/Gulso/Documents/Gul/CredBlock-FL/data/organizations/organization=Org1/*.parquet


In [5]:
org1_users = duckdb.sql(f"""
    SELECT
        "User ID",
        ROW_NUMBER() OVER (
            ORDER BY "User ID"
        ) - 1 AS user_node_id

    FROM (
        SELECT DISTINCT
            "User ID"

        FROM read_parquet(
            '{ORG1_PATH}'
        )
    )

    ORDER BY user_node_id
""").df()

print(org1_users.shape)
display(org1_users.head())

(791567, 2)


,User ID,user_node_id
0,-9223371191532286299,0
1,-9223351024921207274,1
2,-9223320431696609944,2
3,-9223311529942795212,3
4,-9223287066183308537,4


In [6]:
org1_ips = duckdb.sql(f"""
    SELECT
        "IP Address",
        ROW_NUMBER() OVER (
            ORDER BY "IP Address"
        ) - 1 AS ip_node_id

    FROM (
        SELECT DISTINCT
            "IP Address"

        FROM read_parquet(
            '{ORG1_PATH}'
        )
    )

    ORDER BY ip_node_id
""").df()

print(org1_ips.shape)
display(org1_ips.head())

(733381, 2)


,IP Address,ip_node_id
0,10.0.0.1,0
1,10.0.0.10,1
2,10.0.0.100,2
3,10.0.0.101,3
4,10.0.0.102,4


In [7]:
USER_COUNT = len(org1_users)

org1_ips["global_ip_node_id"] = (
    org1_ips["ip_node_id"] + USER_COUNT
)

display(org1_ips.head())

print("User node ID range:")
print(org1_users["user_node_id"].min(), "to",
      org1_users["user_node_id"].max())

print("\nIP node ID range:")
print(org1_ips["global_ip_node_id"].min(), "to",
      org1_ips["global_ip_node_id"].max())

,IP Address,ip_node_id,global_ip_node_id
0,10.0.0.1,0,791567
1,10.0.0.10,1,791568
2,10.0.0.100,2,791569
3,10.0.0.101,3,791570
4,10.0.0.102,4,791571


User node ID range:
0 to 791566

IP node ID range:
791567 to 1524947


In [8]:
org1_edges = duckdb.sql(f"""
    WITH unique_pairs AS (
        SELECT DISTINCT
            "User ID",
            "IP Address"

        FROM read_parquet(
            '{ORG1_PATH}'
        )
    )

    SELECT
        u.user_node_id AS user_node_id,
        i.global_ip_node_id AS ip_node_id

    FROM unique_pairs p

    INNER JOIN org1_users u
        ON p."User ID" = u."User ID"

    INNER JOIN org1_ips i
        ON p."IP Address" = i."IP Address"
""").df()

print(org1_edges.shape)
display(org1_edges.head())

(1697608, 2)


,user_node_id,ip_node_id
0,780976,812976
1,211936,815654
2,436281,791693
3,293617,803542
4,211936,815810


In [9]:
print("Total edges:", len(org1_edges))

print("\nDuplicate edges:")
print(
    org1_edges.duplicated(
        subset=["user_node_id", "ip_node_id"]
    ).sum()
)

print("\nMissing values:")
print(org1_edges.isnull().sum())

print("\nUser node ID range in edges:")
print(
    org1_edges["user_node_id"].min(),
    "to",
    org1_edges["user_node_id"].max()
)

print("\nIP node ID range in edges:")
print(
    org1_edges["ip_node_id"].min(),
    "to",
    org1_edges["ip_node_id"].max()
)

print("\nUnique users represented in edges:")
print(org1_edges["user_node_id"].nunique())

print("\nUnique IPs represented in edges:")
print(org1_edges["ip_node_id"].nunique())

Total edges: 1697608

Duplicate edges:
0

Missing values:
user_node_id    0
ip_node_id      0
dtype: int64

User node ID range in edges:
0 to 791566

IP node ID range in edges:
791567 to 1524947

Unique users represented in edges:
791567

Unique IPs represented in edges:
733381


In [10]:
org1_edge_features = duckdb.sql(f"""
    WITH pair_features AS (
        SELECT
            "User ID",
            "IP Address",

            COUNT(*) AS total_interactions,

            SUM(
                CASE WHEN "Login Successful" = TRUE
                THEN 1 ELSE 0 END
            ) AS successful_interactions,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_interactions,

            AVG(
                CASE WHEN "Login Successful" = FALSE
                THEN 1.0 ELSE 0.0 END
            ) AS failure_ratio,

            MIN("Login Timestamp") AS first_seen,

            MAX("Login Timestamp") AS last_seen

        FROM read_parquet(
            '{ORG1_PATH}'
        )

        GROUP BY
            "User ID",
            "IP Address"
    )

    SELECT
        u.user_node_id,
        i.global_ip_node_id AS ip_node_id,

        p.total_interactions,
        p.successful_interactions,
        p.failed_interactions,
        p.failure_ratio,
        p.first_seen,
        p.last_seen

    FROM pair_features p

    INNER JOIN org1_users u
        ON p."User ID" = u."User ID"

    INNER JOIN org1_ips i
        ON p."IP Address" = i."IP Address"
""").df()

print(org1_edge_features.shape)
display(org1_edge_features.head())

(1697608, 8)


,user_node_id,ip_node_id,total_interactions,successful_interactions,failed_interactions,failure_ratio,first_seen,last_seen
0,620056,796178,1,1.0,0.0,0.000000,2020-02-04 07:35:38.590,2020-02-04 07:35:38.590
1,218471,1197474,14,12.0,2.0,0.142857,2020-02-04 07:41:12.925,2020-12-14 21:04:57.932
2,191950,806893,8,8.0,0.0,0.000000,2020-02-04 07:46:46.846,2020-11-23 11:02:00.988
3,308477,1220129,1,0.0,1.0,1.000000,2020-02-04 08:18:07.462,2020-02-04 08:18:07.462
4,616396,1405832,1,1.0,0.0,0.000000,2020-02-04 08:20:24.748,2020-02-04 08:20:24.748


In [11]:
print("Number of unique edges:")
print(len(org1_edge_features))

print("\nTotal interactions represented:")
print(org1_edge_features["total_interactions"].sum())

print("\nSuccessful interactions:")
print(org1_edge_features["successful_interactions"].sum())

print("\nFailed interactions:")
print(org1_edge_features["failed_interactions"].sum())

print("\nSuccess + Failed:")
print(
    org1_edge_features["successful_interactions"].sum()
    + org1_edge_features["failed_interactions"].sum()
)

print("\nMissing values:")
print(org1_edge_features.isnull().sum())

print("\nInvalid timestamp order:")
print(
    (
        org1_edge_features["first_seen"]
        > org1_edge_features["last_seen"]
    ).sum()
)

Number of unique edges:
1697608

Total interactions represented:
3080150

Successful interactions:
2219740.0

Failed interactions:
860410.0

Success + Failed:
3080150.0

Missing values:
user_node_id               0
ip_node_id                 0
total_interactions         0
successful_interactions    0
failed_interactions        0
failure_ratio              0
first_seen                 0
last_seen                  0
dtype: int64

Invalid timestamp order:
0


In [12]:
org1_temporal_edge_features = duckdb.sql(f"""
    WITH pair_windows AS (
        SELECT
            "User ID",
            "IP Address",

            time_bucket(
                INTERVAL '10 minutes',
                "Login Timestamp"
            ) AS ten_minute_window,

            COUNT(*) AS interactions_in_window,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_in_window

        FROM read_parquet(
            '{ORG1_PATH}'
        )

        GROUP BY
            "User ID",
            "IP Address",
            ten_minute_window
    ),

    pair_temporal AS (
        SELECT
            "User ID",
            "IP Address",

            COUNT(*) AS active_10min_windows,

            MAX(interactions_in_window)
                AS max_interactions_10min,

            MAX(failed_in_window)
                AS max_failed_interactions_10min

        FROM pair_windows

        GROUP BY
            "User ID",
            "IP Address"
    )

    SELECT
        u.user_node_id,
        i.global_ip_node_id AS ip_node_id,

        p.active_10min_windows,
        p.max_interactions_10min,
        p.max_failed_interactions_10min

    FROM pair_temporal p

    INNER JOIN org1_users u
        ON p."User ID" = u."User ID"

    INNER JOIN org1_ips i
        ON p."IP Address" = i."IP Address"
""").df()

print(org1_temporal_edge_features.shape)
display(org1_temporal_edge_features.head())

(1697608, 5)


,user_node_id,ip_node_id,active_10min_windows,max_interactions_10min,max_failed_interactions_10min
0,147113,1330086,2,1,0.0
1,16649,1329994,14,1,1.0
2,774095,991271,1,1,1.0
3,424829,1398997,38,2,1.0
4,480786,800502,1,1,0.0


In [13]:
org1_final_edges = org1_edge_features.merge(
    org1_temporal_edge_features,
    on=["user_node_id", "ip_node_id"],
    how="inner",
    validate="one_to_one"
)

# Convert the first/last timestamps into a numeric duration feature
org1_final_edges["relationship_duration_hours"] = (
    (
        org1_final_edges["last_seen"]
        - org1_final_edges["first_seen"]
    ).dt.total_seconds()
    / 3600
)

print(org1_final_edges.shape)

display(org1_final_edges.head())

(1697608, 12)


,user_node_id,ip_node_id,total_interactions,successful_interactions,failed_interactions,failure_ratio,first_seen,last_seen,active_10min_windows,max_interactions_10min,max_failed_interactions_10min,relationship_duration_hours
0,620056,796178,1,1.0,0.0,0.000000,2020-02-04 07:35:38.590,2020-02-04 07:35:38.590,1,1,0.0,0.000000
1,218471,1197474,14,12.0,2.0,0.142857,2020-02-04 07:41:12.925,2020-12-14 21:04:57.932,13,2,1.0,7549.395835
2,191950,806893,8,8.0,0.0,0.000000,2020-02-04 07:46:46.846,2020-11-23 11:02:00.988,7,2,0.0,7035.253928
3,308477,1220129,1,0.0,1.0,1.000000,2020-02-04 08:18:07.462,2020-02-04 08:18:07.462,1,1,1.0,0.000000
4,616396,1405832,1,1.0,0.0,0.000000,2020-02-04 08:20:24.748,2020-02-04 08:20:24.748,1,1,0.0,0.000000


In [14]:
org1_user_features = duckdb.sql(f"""
    WITH user_basic AS (
        SELECT
            "User ID",

            COUNT(*) AS total_logins,

            SUM(
                CASE WHEN "Login Successful" = TRUE
                THEN 1 ELSE 0 END
            ) AS successful_logins,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_logins,

            AVG(
                CASE WHEN "Login Successful" = FALSE
                THEN 1.0 ELSE 0.0 END
            ) AS failure_ratio,

            COUNT(DISTINCT "IP Address")
                AS distinct_ips

        FROM read_parquet(
            '{ORG1_PATH}'
        )

        GROUP BY "User ID"
    ),

    user_windows AS (
        SELECT
            "User ID",

            time_bucket(
                INTERVAL '10 minutes',
                "Login Timestamp"
            ) AS ten_minute_window,

            COUNT(*) AS logins_in_window,

            COUNT(DISTINCT "IP Address")
                AS distinct_ips_in_window

        FROM read_parquet(
            '{ORG1_PATH}'
        )

        GROUP BY
            "User ID",
            ten_minute_window
    ),

    temporal_summary AS (
        SELECT
            "User ID",

            MAX(logins_in_window)
                AS max_logins_10min,

            MAX(distinct_ips_in_window)
                AS max_distinct_ips_10min

        FROM user_windows

        GROUP BY "User ID"
    )

    SELECT
        u.user_node_id,

        b.total_logins,
        b.successful_logins,
        b.failed_logins,
        b.failure_ratio,
        b.distinct_ips,

        t.max_logins_10min,
        t.max_distinct_ips_10min

    FROM user_basic b

    INNER JOIN temporal_summary t
        ON b."User ID" = t."User ID"

    INNER JOIN org1_users u
        ON b."User ID" = u."User ID"

    ORDER BY u.user_node_id
""").df()

print(org1_user_features.shape)
display(org1_user_features.head())

(791567, 8)


,user_node_id,total_logins,successful_logins,failed_logins,failure_ratio,distinct_ips,max_logins_10min,max_distinct_ips_10min
0,0,1,0.0,1.0,1.000000,1,1,1
1,1,3,2.0,1.0,0.333333,2,2,1
2,2,1,0.0,1.0,1.000000,1,1,1
3,3,1,1.0,0.0,0.000000,1,1,1
4,4,10,7.0,3.0,0.300000,1,2,1


In [15]:
print("Total user nodes:")
print(len(org1_user_features))

print("\nUnique user node IDs:")
print(org1_user_features["user_node_id"].nunique())

print("\nMissing values:")
print(org1_user_features.isnull().sum())

print("\nTotal logins represented:")
print(org1_user_features["total_logins"].sum())

print("\nSuccessful logins:")
print(org1_user_features["successful_logins"].sum())

print("\nFailed logins:")
print(org1_user_features["failed_logins"].sum())

print("\nInvalid failure ratios:")
print(
    (
        (org1_user_features["failure_ratio"] < 0)
        | (org1_user_features["failure_ratio"] > 1)
    ).sum()
)

print("\nInvalid temporal values:")
print(
    (
        org1_user_features["max_logins_10min"]
        > org1_user_features["total_logins"]
    ).sum()
)

print("\nInvalid IP diversity values:")
print(
    (
        org1_user_features["max_distinct_ips_10min"]
        > org1_user_features["distinct_ips"]
    ).sum()
)

Total user nodes:
791567

Unique user node IDs:
791567

Missing values:
user_node_id              0
total_logins              0
successful_logins         0
failed_logins             0
failure_ratio             0
distinct_ips              0
max_logins_10min          0
max_distinct_ips_10min    0
dtype: int64

Total logins represented:
3080150

Successful logins:
2219740.0

Failed logins:
860410.0

Invalid failure ratios:
0

Invalid temporal values:
0

Invalid IP diversity values:
0


In [16]:
org1_ip_features = duckdb.sql(f"""
    WITH ip_basic AS (
        SELECT
            "IP Address",

            COUNT(*) AS total_logins,

            SUM(
                CASE WHEN "Login Successful" = TRUE
                THEN 1 ELSE 0 END
            ) AS successful_logins,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_logins,

            AVG(
                CASE WHEN "Login Successful" = FALSE
                THEN 1.0 ELSE 0.0 END
            ) AS failure_ratio,

            COUNT(DISTINCT "User ID")
                AS distinct_users

        FROM read_parquet(
            '{ORG1_PATH}'
        )

        GROUP BY "IP Address"
    ),

    ip_windows AS (
        SELECT
            "IP Address",

            time_bucket(
                INTERVAL '10 minutes',
                "Login Timestamp"
            ) AS ten_minute_window,

            COUNT(*) AS logins_in_window,

            COUNT(DISTINCT "User ID")
                AS distinct_users_in_window

        FROM read_parquet(
            '{ORG1_PATH}'
        )

        GROUP BY
            "IP Address",
            ten_minute_window
    ),

    temporal_summary AS (
        SELECT
            "IP Address",

            MAX(logins_in_window)
                AS max_logins_10min,

            MAX(distinct_users_in_window)
                AS max_distinct_users_10min

        FROM ip_windows

        GROUP BY "IP Address"
    )

    SELECT
        i.global_ip_node_id AS ip_node_id,

        b.total_logins,
        b.successful_logins,
        b.failed_logins,
        b.failure_ratio,
        b.distinct_users,

        t.max_logins_10min,
        t.max_distinct_users_10min

    FROM ip_basic b

    INNER JOIN temporal_summary t
        ON b."IP Address" = t."IP Address"

    INNER JOIN org1_ips i
        ON b."IP Address" = i."IP Address"

    ORDER BY i.global_ip_node_id
""").df()

print(org1_ip_features.shape)
display(org1_ip_features.head())

(733381, 8)


,ip_node_id,total_logins,successful_logins,failed_logins,failure_ratio,distinct_users,max_logins_10min,max_distinct_users_10min
0,791567,25,16.0,9.0,0.360000,22,2,1
1,791568,22,19.0,3.0,0.136364,15,3,1
2,791569,92,52.0,40.0,0.434783,68,3,1
3,791570,67,33.0,34.0,0.507463,49,3,1
4,791571,88,41.0,47.0,0.534091,65,4,2


In [17]:
print("Total IP nodes:")
print(len(org1_ip_features))

print("\nUnique IP node IDs:")
print(org1_ip_features["ip_node_id"].nunique())

print("\nMissing values:")
print(org1_ip_features.isnull().sum())

print("\nTotal logins represented:")
print(org1_ip_features["total_logins"].sum())

print("\nSuccessful logins:")
print(org1_ip_features["successful_logins"].sum())

print("\nFailed logins:")
print(org1_ip_features["failed_logins"].sum())

print("\nInvalid failure ratios:")
print(
    (
        (org1_ip_features["failure_ratio"] < 0)
        | (org1_ip_features["failure_ratio"] > 1)
    ).sum()
)

print("\nInvalid temporal values:")
print(
    (
        org1_ip_features["max_logins_10min"]
        > org1_ip_features["total_logins"]
    ).sum()
)

print("\nInvalid user diversity values:")
print(
    (
        org1_ip_features["max_distinct_users_10min"]
        > org1_ip_features["distinct_users"]
    ).sum()
)

print("\nIP node ID range:")
print(
    org1_ip_features["ip_node_id"].min(),
    "to",
    org1_ip_features["ip_node_id"].max()
)

Total IP nodes:
733381

Unique IP node IDs:
733381

Missing values:
ip_node_id                  0
total_logins                0
successful_logins           0
failed_logins               0
failure_ratio               0
distinct_users              0
max_logins_10min            0
max_distinct_users_10min    0
dtype: int64

Total logins represented:
3080150

Successful logins:
2219740.0

Failed logins:
860410.0

Invalid failure ratios:
0

Invalid temporal values:
0

Invalid user diversity values:
0

IP node ID range:
791567 to 1524947


In [18]:
import pandas as pd

user_node_table = org1_user_features.rename(
    columns={
        "distinct_ips": "distinct_neighbors",
        "max_distinct_ips_10min": "max_distinct_neighbors_10min"
    }
).copy()

user_node_table["node_type"] = 0   # 0 = User


ip_node_table = org1_ip_features.rename(
    columns={
        "ip_node_id": "node_id",
        "distinct_users": "distinct_neighbors",
        "max_distinct_users_10min": "max_distinct_neighbors_10min"
    }
).copy()

ip_node_table["node_type"] = 1     # 1 = IP


user_node_table = user_node_table.rename(
    columns={"user_node_id": "node_id"}
)


org1_node_features = pd.concat(
    [user_node_table, ip_node_table],
    ignore_index=True
).sort_values("node_id").reset_index(drop=True)


print(org1_node_features.shape)
display(org1_node_features.head())
display(org1_node_features.tail())

(1524948, 9)


,node_id,total_logins,successful_logins,failed_logins,failure_ratio,distinct_neighbors,max_logins_10min,max_distinct_neighbors_10min,node_type
0,0,1,0.0,1.0,1.000000,1,1,1,0
1,1,3,2.0,1.0,0.333333,2,2,1,0
2,2,1,0.0,1.0,1.000000,1,1,1,0
3,3,1,1.0,0.0,0.000000,1,1,1,0
4,4,10,7.0,3.0,0.300000,1,2,1,0


,node_id,total_logins,successful_logins,failed_logins,failure_ratio,distinct_neighbors,max_logins_10min,max_distinct_neighbors_10min,node_type
1524943,1524943,1,1.0,0.0,0.000000,1,1,1,1
1524944,1524944,3,2.0,1.0,0.333333,1,2,1,1
1524945,1524945,1,1.0,0.0,0.000000,1,1,1,1
1524946,1524946,1,1.0,0.0,0.000000,1,1,1,1
1524947,1524947,1,0.0,1.0,1.000000,1,1,1,1


In [19]:
print("Total nodes:")
print(len(org1_node_features))

print("\nUnique node IDs:")
print(org1_node_features["node_id"].nunique())

print("\nNode ID range:")
print(
    org1_node_features["node_id"].min(),
    "to",
    org1_node_features["node_id"].max()
)

print("\nMissing values:")
print(org1_node_features.isnull().sum())

print("\nNode type counts:")
print(org1_node_features["node_type"].value_counts().sort_index())

print("\nNode IDs correctly ordered:")
print(
    (
        org1_node_features["node_id"].to_numpy()
        == range(len(org1_node_features))
    ).all()
)

Total nodes:
1524948

Unique node IDs:
1524948

Node ID range:
0 to 1524947

Missing values:
node_id                         0
total_logins                    0
successful_logins               0
failed_logins                   0
failure_ratio                   0
distinct_neighbors              0
max_logins_10min                0
max_distinct_neighbors_10min    0
node_type                       0
dtype: int64

Node type counts:
node_type
0    791567
1    733381
Name: count, dtype: int64

Node IDs correctly ordered:
True
